In [30]:
df = spark.read.format('delta').load("abfss://TaxiTrips@onelake.dfs.fabric.microsoft.com/BronzeLayer.Lakehouse/Files/taxi_trips")
# df now is a Spark DataFrame containing JSON data from "Files/2026-04-16_taxitrips/data_fdbbedb0-bc8f-406b-a988-b077a735caef_3e9770b5-c231-409e-a70d-44a106ab952e.json".

StatementMeta(, 17bc12e1-6992-4ae5-b448-8f21e36cefbc, 32, Finished, Available, Finished, False)

In [31]:
df.write.format('delta').mode('overwrite').saveAsTable('BronzeLayer.trips.trips_table')

StatementMeta(, 17bc12e1-6992-4ae5-b448-8f21e36cefbc, 33, Finished, Available, Finished, False)

In [32]:
display(df)

StatementMeta(, 17bc12e1-6992-4ae5-b448-8f21e36cefbc, 34, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 97f79cca-bd2c-415d-9d74-d4daf485c1d8)

In [33]:
if spark.catalog.tableExists("SilverLayer.trips.trips_data"):
    last_load_date = spark.sql("SELECT max(trip_start_timestamp) FROM SilverLayer.trips.trips_data").collect()[0][0]
else:
    last_load_date = '1000-01-01'

StatementMeta(, 17bc12e1-6992-4ae5-b448-8f21e36cefbc, 35, Finished, Available, Finished, False)

In [34]:
df_bronze_source =spark.sql(f"""
            SELECT * FROM BronzeLayer.trips.trips_table
            WHERE CAST(trip_start_timestamp AS TIMESTAMP) >  TIMESTAMP('{last_load_date}')
            """
        )

StatementMeta(, 17bc12e1-6992-4ae5-b448-8f21e36cefbc, 36, Finished, Available, Finished, False)

**DROP COLUMNS LOCATION COLUMNS FOR PICKUP AND DROPOFF**

In [9]:
# df = df.drop('dropoff_centroid_location','pickup_centroid_location')

StatementMeta(, 236e5f4c-1b9e-48e0-8439-01c76c5c4ce3, 11, Finished, Available, Finished, False)

**RENAMED COLUMNS**

In [35]:
df =df_bronze_source

StatementMeta(, 17bc12e1-6992-4ae5-b448-8f21e36cefbc, 37, Finished, Available, Finished, False)

In [15]:
df = df.withColumnRenamed('dropoff_centroid_latitude', 'dropoff_lat') \
        .withColumnRenamed('dropoff_centroid_longitude', 'dropoff_long') \
        .withColumnRenamed('pickup_centroid_latitude', 'pickup_lat') \
        .withColumnRenamed('pickup_centroid_longitude', 'pickup_long')


StatementMeta(, 236e5f4c-1b9e-48e0-8439-01c76c5c4ce3, 17, Finished, Available, Finished, False)

**SCHEMA ENFORCEMENT ON DATAFRAME**

In [17]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

df = df.withColumn('dropoff_lat', col('dropoff_lat').cast(DoubleType())) \
        .withColumn('dropoff_long', col('dropoff_long').cast(DoubleType())) \
        .withColumn('extras', col('extras').cast(DoubleType())) \
        .withColumn('fare',col('fare').cast(DoubleType())) \
        .withColumn('pickup_lat', col('pickup_lat').cast(DoubleType())) \
        .withColumn('pickup_long', col('pickup_long').cast(S)) \
        .withColumn('tips', col('tips').cast(DoubleType())) \
        .withColumn('tolls',col('tolls').cast(DoubleType())) \
        .withColumn('trip_end_timestamp', col('trip_end_timestamp').cast(TimestampType())) \
        .withColumn('trip_start_timestamp', col('trip_start_timestamp').cast(TimestampType())) \
        .withColumn('trip_miles', col('trip_miles').cast(DoubleType())) \
        .withColumn('trip_seconds',col('trip_seconds').cast(IntegerType())) \
        .withColumn('trip_total',col('trip_total').cast(DoubleType())) 


StatementMeta(, 236e5f4c-1b9e-48e0-8439-01c76c5c4ce3, 19, Finished, Available, Finished, False)

**Handling Null and Missing Values in DF**

In [18]:
nulls_dict = df.select([(sum(col(c).isNull().cast('int')).alias(c)) for c in df.columns]).collect()[0].asDict()



cols_keep = []
for key,value in nulls_dict.items():
    if value < 100000:
        cols_keep.append(key)


df = df.select(cols_keep)



StatementMeta(, 236e5f4c-1b9e-48e0-8439-01c76c5c4ce3, 20, Finished, Available, Finished, False)

In [19]:
print(nulls_dict)

StatementMeta(, 236e5f4c-1b9e-48e0-8439-01c76c5c4ce3, 21, Finished, Available, Finished, False)

{'company': 0, 'dropoff_lat': 1241349, 'dropoff_long': 1241349, 'extras': 31160, 'fare': 31160, 'payment_type': 0, 'pickup_lat': 405816, 'pickup_long': 405816, 'tips': 31160, 'tolls': 31160, 'trip_end_timestamp': 329, 'trip_id': 0, 'trip_miles': 123, 'trip_seconds': 2847, 'trip_start_timestamp': 0, 'trip_total': 31160}


In [20]:

numeric_cols = [c for c, t in df.dtypes if t in ("int", "double")]
df = df.fillna(0.0, subset=numeric_cols)


StatementMeta(, 236e5f4c-1b9e-48e0-8439-01c76c5c4ce3, 22, Finished, Available, Finished, False)

In [21]:
string_cols = [c for c,t in df.dtypes if t in ("string")]

df = df.fillna('UNKNOWN',subset=string_cols)




StatementMeta(, 236e5f4c-1b9e-48e0-8439-01c76c5c4ce3, 23, Finished, Available, Finished, False)

In [22]:
from pyspark.sql.functions import col, avg


avg_trip_time = df.select(
    round(avg(col("trip_seconds")/60),0).alias("avg_trip_seconds")).collect()[0][0]


print(avg_trip_time)


StatementMeta(, 236e5f4c-1b9e-48e0-8439-01c76c5c4ce3, 24, Finished, Available, Finished, False)

20.0


In [23]:
df = df.withColumn('trip_end_timestamp',when(
            col('trip_end_timestamp').isNull(),expr(f"trip_start_timestamp + INTERVAL  {int(avg_trip_time)} MINUTES")).otherwise(col('trip_end_timestamp')))

StatementMeta(, 236e5f4c-1b9e-48e0-8439-01c76c5c4ce3, 25, Finished, Available, Finished, False)

In [24]:
display(df)

StatementMeta(, 236e5f4c-1b9e-48e0-8439-01c76c5c4ce3, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7ddc7d14-c8c8-4306-8533-001a883a9a17)

In [ ]:
df.createOrReplaceTempView('siver_source')

In [ ]:
%%sql
MERGE INTO SilverLayer.trips.trips_data
USING silver_source
ON SilverLayer.trips.trips_data.trip_id = silver_source.trip_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [26]:
# df.write.format('delta').mode('overwrite') \
#         .option("path", "abfss://TaxiTrips@onelake.dfs.fabric.microsoft.com/SilverLayer.Lakehouse/Files/Taxitrips") \
#         .saveAsTable("SilverLayer.trips.trips_data")

StatementMeta(, 236e5f4c-1b9e-48e0-8439-01c76c5c4ce3, 28, Finished, Available, Finished, False)